In [0]:
-- Question 1: What is the unique count and total amount for each transaction type?
SELECT txn_type,
count(txn_type),
sum(txn_amount)
FROM customer_transactions
GROUP BY txn_type;

-- Question 2: What is the average total historical deposit counts and amounts for all customers?
WITH cte as (SELECT 
customer_id,
count(*) as historical_deposits,
avg(txn_amount) as avg_deposit
FROM customer_transactions
WHERE txn_type = 'deposit'
GROUP BY customer_id)

SELECT 
ROUND(avg(historical_deposits), 0) as avg_count,
ROUND(avg(avg_deposit), 2) as avg_amount
FROM cte;


-- Question 3: For each month - how many Data Bank customers make more than 1 deposit and either 1 purchase or 1 withdrawal in a single month?

WITH monthly_data as (SELECT 
customer_id,
to_date(date_trunc('month', txn_date)) as month,
SUM(CASE WHEN txn_type = 'deposit' THEN 1 ELSE 0 END) as deposit,
SUM(CASE WHEN txn_type = 'purchase' THEN 1 ELSE 0 END) as purchase,
SUM(CASE WHEN txn_type = 'withdrawal' THEN 1 ELSE 0 END) as withdrawal
FROM customer_transactions
GROUP BY customer_id, to_date(date_trunc('month', txn_date)))

SELECT month,
count(*) as customers
FROM monthly_data
WHERE deposit > 1 AND (purchase >= 1 OR withdrawal >= 1)
GROUP BY month
ORDER BY month;


-- Question 4: What is the closing balance for each customer at the end of the month?

WITH cte AS (
    SELECT 
        customer_id, 
        txn_date,
        date_trunc('month', txn_date)::date AS month,
        CASE 
            WHEN txn_type = 'deposit' THEN txn_amount
            WHEN txn_type IN ('purchase', 'withdrawal') THEN -txn_amount
            ELSE 0
        END AS signed_amount
    FROM customer_transactions
),
running AS (
    SELECT
        customer_id,
        month,
        txn_date,
        SUM(signed_amount) OVER (
            PARTITION BY customer_id
            ORDER BY txn_date
        ) AS running_balance
    FROM cte
),
end_of_month AS (
    SELECT
        customer_id,
        month,
        MAX(txn_date) AS month_end_txn
    FROM running
    GROUP BY customer_id, month
)
SELECT
    r.customer_id,
    r.month,
    r.running_balance AS closing_balance
FROM running r
JOIN end_of_month e
  ON r.customer_id = e.customer_id
 AND r.month = e.month
 AND r.txn_date = e.month_end_txn
ORDER BY r.customer_id, r.month;


-- Question 5: What is the percentage of customers who increase their closing balance by more than 5%?

with cte as (SELECT
customer_id,
txn_date,
date_trunc('month', txn_date)::date AS month,
CASE WHEN txn_type = 'deposit' THEN txn_amount
     WHEN txn_type IN ('purchase', 'withdrawal') THEN -txn_amount
     ELSE 0 END as signed_amount
FROM customer_transactions),

running_total as (
SELECT customer_id,
month,
txn_date,
signed_amount,
SUM(signed_amount) OVER (PARTITION BY customer_id ORDER BY txn_date) AS running_balance
FROM cte),

end_of_month as (
SELECT customer_id,
month,
MAX(txn_date) AS month_end_txn
FROM running_total
GROUP BY customer_id, month), 

closing_balance as (
SELECT r.customer_id,
r.month,
r.running_balance as closing_balance
FROM running_total r
JOIN end_of_month e ON 
r.customer_id = e.customer_id
AND r.month = e.month 
AND r.txn_date = e.month_end_txn
ORDER BY r.customer_id, r.month), 

balance_changes as(
SELECT *,
LAG(closing_balance) OVER (PARTITION BY customer_id ORDER BY month) AS previous_balance 
FROM closing_balance),

customer_flags as (SELECT customer_id,
MAX(CASE WHEN closing_balance > previous_balance 
         AND previous_balance IS NOT null
         AND previous_balance <> 0
         AND closing_balance / previous_balance > 1.05 THEN 1 ELSE 0 END) 
         AS increased_gt_5_pct
FROM balance_changes
GROUP BY customer_id)

SELECT 
(SUM(increased_gt_5_pct)
/ COUNT(*) ) * 100 as pct_customers_gt_5_pct
FROM customer_flags ;













